# 11_developer_reputation.ipynb

**Experimento: Developer Reputation como Feature**

Testa si agregar la reputación del desarrollador (promedio histórico de reviews de sus juegos pre-2016) 
mejora la predicción para juegos post-2016.

## Modelos:
- **11a**: RS embeddings (64d) + dev_rep (1d) — vs baseline modelo 03
- **11b**: Hybrid (RS 64d + TF-IDF 100d + Numeric 2d) + dev_rep (1d) — vs baseline modelo 08

## Hipótesis:
Para juegos post-2016 (donde los embeddings RS son aleatorios/no entrenados),
la reputación del desarrollador puede actuar como proxy de popularidad futura.

## Input:
- `../Data/item_embeddings_rs_clean.npy` — embeddings RS entrenados
- `../Data/developer_reputation.npy` — reputación por item_idx

## Output:
- Resultados guardados en `experiment_results.json` con IDs 11a y 11b

In [1]:
import pandas as pd
import numpy as np
import json, ast, sys, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

INTERACTIONS = "../Data/interactions.parquet"
USERMAP      = "../Data/user2idx.json"
ITEMMAP      = "../Data/item2idx.json"
ITEM_EMB     = "../Data/item_embeddings_rs_clean.npy"
DEV_REP      = "../Data/developer_reputation.npy"
STEAM_GAMES  = "../Data/steam_games.json"
CUTOFF       = pd.to_datetime('2016-01-01')

df_inter = pd.read_parquet(INTERACTIONS)
item_emb = np.load(ITEM_EMB)   # (3682, 64)
dev_rep  = np.load(DEV_REP)    # (3682,) indexed by item_idx

with open(USERMAP, "r") as f: user2idx = {k:int(v) for k,v in json.load(f).items()}
with open(ITEMMAP, "r") as f: item2idx = {k:int(v) for k,v in json.load(f).items()}

target_df = df_inter.groupby("item_idx").size().reset_index(name="total_reviews")

games = []
with open(STEAM_GAMES, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try:
            games.append(ast.literal_eval(line))
        except Exception:
            pass

df_games = pd.json_normalize(games)
df_games = df_games.rename(columns={"id": "item_id"})
df_games["item_idx"] = df_games["item_id"].map(item2idx)
df_games = df_games.dropna(subset=["item_idx"])
df_games["item_idx"] = df_games["item_idx"].astype(int)
df_games['release_date_parsed'] = pd.to_datetime(df_games['release_date'], errors='coerce')

print(f"RS embeddings shape:   {item_emb.shape}")
print(f"Dev rep shape:         {dev_rep.shape}")
print(f"Games with dev rep>0:  {(dev_rep > 0).sum()}")
print(f"Dev rep stats - mean={dev_rep[dev_rep>0].mean():.1f}, max={dev_rep.max():.1f}")

C:\Users\matia\anaconda3\envs\VG_RS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RS embeddings shape:   (3682, 64)
Dev rep shape:         (3682,)
Games with dev rep>0:  2217
Dev rep stats - mean=20.4, max=1881.0


In [2]:
TSCV_WINDOWS = [
    ('2013-07-01', '2014-01-01'),
    ('2014-01-01', '2014-07-01'),
    ('2014-07-01', '2015-01-01'),
    ('2015-01-01', '2015-07-01'),
    ('2015-07-01', '2016-01-01'),
]

def run_experiment(X_feat, y, data_df, model_id, model_name, features_desc, emb_type):
    """Full experiment: Optuna tuning + temporal + TSCV + KFold + save."""
    valid_mask = data_df['release_date_parsed'].notna().values
    train_mask = valid_mask & (data_df['release_date_parsed'] < CUTOFF).values
    test_mask  = valid_mask & (data_df['release_date_parsed'] >= CUTOFF).values

    X_tr, X_te = X_feat[train_mask], X_feat[test_mask]
    y_tr, y_te = y[train_mask], y[test_mask]

    print(f"\n{'='*70}")
    print(f"EXPERIMENT [{model_id}]: {model_name}")
    print(f"Features: {features_desc}  |  X shape: {X_feat.shape}")
    print(f"Train: {len(X_tr)}, Test: {len(X_te)}")
    print(f"{'='*70}")

    # Optuna tuning
    # Sort by release date so Optuna val set is always the most recent 20%
    _sort_order = np.argsort(data_df.loc[train_mask, 'release_date_parsed'].values)
    X_tr = X_tr[_sort_order]
    y_tr = y_tr[_sort_order]

    n_tr = len(X_tr)
    split_idx = max(10, int(n_tr * 0.8))
    X_opt, X_val = X_tr[:split_idx], X_tr[split_idx:]
    y_opt, y_val = y_tr[:split_idx], y_tr[split_idx:]

    def objective(trial):
        p = dict(
            n_estimators=trial.suggest_int('n_estimators', 100, 800),
            learning_rate=trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            max_depth=trial.suggest_int('max_depth', 3, 8),
            min_child_weight=trial.suggest_int('min_child_weight', 1, 10),
            subsample=trial.suggest_float('subsample', 0.5, 1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
            gamma=trial.suggest_float('gamma', 0.0, 2.0),
            random_state=42, tree_method='hist', verbosity=0,
        )
        m = XGBRegressor(**p)
        m.fit(X_opt, y_opt)
        return float(mean_squared_error(y_val, m.predict(X_val)) ** 0.5)

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=50, show_progress_bar=True)
    best_params = {**study.best_params, 'random_state': 42, 'tree_method': 'hist', 'verbosity': 0}
    print(f"Best val RMSE: {study.best_value:.4f}")

    # Temporal test
    model = XGBRegressor(**best_params)
    model.fit(X_tr, y_tr)
    pred_te = model.predict(X_te)

    r2_temporal   = r2_score(y_te, pred_te)
    rmse_temporal = mean_squared_error(y_te, pred_te) ** 0.5
    mae_temporal  = mean_absolute_error(y_te, pred_te)
    mape_temporal = mean_absolute_percentage_error(y_te, pred_te)
    print(f"Temporal: R²={r2_temporal:.6f} | RMSE={rmse_temporal:.2f} | MAE={mae_temporal:.2f}")

    # TSCV
    item_dates = data_df['release_date_parsed'].values
    ts_results = []
    for tc_str, sc_str in TSCV_WINDOWS:
        tc, sc = pd.Timestamp(tc_str), pd.Timestamp(sc_str)
        t_msk = np.array([pd.notna(d) and d < tc for d in item_dates])
        e_msk = np.array([pd.notna(d) and d >= tc and d < sc for d in item_dates])
        if e_msk.sum() < 5: continue
        m_ts = XGBRegressor(**best_params)
        m_ts.fit(X_feat[t_msk], y[t_msk])
        pred_ts = m_ts.predict(X_feat[e_msk])
        ts_results.append({'R2': r2_score(y[e_msk], pred_ts),
                           'RMSE': mean_squared_error(y[e_msk], pred_ts) ** 0.5})
    if ts_results:
        ts_df = pd.DataFrame(ts_results)
        r2_tscv_mean, r2_tscv_std = ts_df['R2'].mean(), ts_df['R2'].std()
        rmse_tscv_mean, rmse_tscv_std = ts_df['RMSE'].mean(), ts_df['RMSE'].std()
    else:
        r2_tscv_mean = r2_tscv_std = rmse_tscv_mean = rmse_tscv_std = float('nan')
    print(f"TSCV:     R²={r2_tscv_mean:.4f} ± {r2_tscv_std:.4f}  RMSE={rmse_tscv_mean:.2f}")

    # KFold
    X_kf = X_feat[valid_mask]
    y_kf = y[valid_mask]
    kf_results = []
    for tr_idx, te_idx in KFold(n_splits=5, shuffle=True, random_state=42).split(X_kf):
        m_kf = XGBRegressor(**best_params)
        m_kf.fit(X_kf[tr_idx], y_kf[tr_idx])
        pred_kf = m_kf.predict(X_kf[te_idx])
        kf_results.append({'R2': r2_score(y_kf[te_idx], pred_kf),
                           'RMSE': mean_squared_error(y_kf[te_idx], pred_kf) ** 0.5})
    kf_df = pd.DataFrame(kf_results)
    r2_kfold_mean, r2_kfold_std = kf_df['R2'].mean(), kf_df['R2'].std()
    rmse_kfold_mean = kf_df['RMSE'].mean()
    print(f"KFold:    R²={r2_kfold_mean:.4f} ± {r2_kfold_std:.4f}  RMSE={rmse_kfold_mean:.2f}")

    # Save
    sys.path.insert(0, os.path.abspath("."))
    from results_tracker import save_result
    save_result(
        model_id=model_id,
        model_name=model_name,
        features=features_desc,
        embeddings=emb_type,
        metrics={
            "r2_temporal":   r2_temporal,
            "rmse_temporal": rmse_temporal,
            "mae_temporal":  mae_temporal,
            "mape_temporal": mape_temporal,
            "r2_tscv":       r2_tscv_mean,
            "r2_tscv_std":   r2_tscv_std,
            "rmse_tscv":     rmse_tscv_mean,
            "rmse_tscv_std": rmse_tscv_std,
            "r2_kfold":      r2_kfold_mean,
            "r2_kfold_std":  r2_kfold_std,
            "rmse_kfold":    rmse_kfold_mean,
        },
    )
    print(f"✅ Saved [{model_id}]")
    return model, r2_temporal, r2_tscv_mean

print("✅ Helper function ready")


✅ Helper function ready


In [3]:
# ── Experiment 11a: RS (64d) + Dev Reputation (1d) ────────────────────────
# Build y and data_df for all 3682 items (same as notebook 03)
all_indices = list(range(len(item_emb)))
y_all = target_df.set_index("item_idx").loc[range(len(item_emb))]["total_reviews"].values

data_df_all = pd.DataFrame({'item_idx': all_indices}).merge(
    df_games[['item_idx', 'release_date_parsed']],
    on='item_idx', how='left'
)

# Add dev_rep as extra column
X_11a = np.hstack([item_emb, dev_rep.reshape(-1, 1)])  # (3682, 65)
print(f"X_11a shape: {X_11a.shape}  (RS 64d + dev_rep 1d)")
print(f"Dev rep coverage in test (post-2016): ", end="")
test_mask_tmp = data_df_all['release_date_parsed'] >= CUTOFF
print(f"{(dev_rep[test_mask_tmp.values] > 0).sum()} / {test_mask_tmp.sum()} games have rep > 0")

model_11a, r2_11a_test, r2_11a_tscv = run_experiment(
    X_11a, y_all, data_df_all,
    model_id="11a",
    model_name="RS + Dev Reputation",
    features_desc="RS clean (64d) + dev_rep (1d)",
    emb_type="clean"
)

X_11a shape: (3682, 65)  (RS 64d + dev_rep 1d)
Dev rep coverage in test (post-2016): 126 / 486 games have rep > 0

EXPERIMENT [11a]: RS + Dev Reputation
Features: RS clean (64d) + dev_rep (1d)  |  X shape: (3682, 65)
Train: 2621, Test: 486


  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: 37.3235:   0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: 37.3235:   2%|▏         | 1/50 [00:00<00:23,  2.07it/s]

Best trial: 1. Best value: 9.85126:   2%|▏         | 1/50 [00:01<00:23,  2.07it/s]

...

Best trial: 41. Best value: 4.20798: 100%|██████████| 50/50 [00:40<00:00,  1.23it/s]

Best val RMSE: 4.2080


Temporal: R²=-0.025799 | RMSE=57.62 | MAE=9.44


TSCV:     R²=0.7779 ± 0.2721  RMSE=20.37


KFold:    R²=0.1019 ± 0.6128  RMSE=71.51
✅ Saved [11a]


In [4]:
# ── Experiment 11b: Hybrid (RS+TF-IDF+Numeric) + Dev Reputation ───────────
# Rebuild hybrid features identical to notebook 08, then append dev_rep

def parse_tags(tags):
    if tags is None or (isinstance(tags, float) and pd.isna(tags)): return []
    if isinstance(tags, list): return [str(t).strip() for t in tags if t]
    if isinstance(tags, str):
        try:
            lst = eval(tags)
            return [str(t).strip() for t in lst if t] if isinstance(lst, list) else []
        except: return []
    return []

def parse_genres(genres):
    if genres is None or (isinstance(genres, float) and pd.isna(genres)): return []
    if isinstance(genres, list): return [str(g).strip() for g in genres if g]
    if isinstance(genres, str):
        try:
            lst = eval(genres)
            return [str(g).strip() for g in lst if g] if isinstance(lst, list) else []
        except: return []
    return []

df_games['tags_list']  = df_games['tags'].apply(parse_tags)
df_games['tag_text']   = df_games['tags_list'].apply(lambda x: ' '.join(x))
df_games['genres_list']= df_games['genres'].apply(parse_genres)
df_games['genre_text'] = df_games['genres_list'].apply(lambda x: ' '.join(x))
df_games['content_text'] = df_games['tag_text'] + ' ' + df_games['genre_text']

def parse_price(p):
    if pd.isna(p) or p == '': return 0.0
    try: return float(p)
    except: return 0.0

df_games['price_num']        = df_games['price'].apply(parse_price)
df_games['early_access_flag']= df_games['early_access'].apply(lambda x: 1 if x else 0)

games_with_content = df_games[df_games['content_text'] != ''].copy()

tfidf = TfidfVectorizer(max_features=100, min_df=2, max_df=0.5, ngram_range=(1, 1))
tfidf_matrix = tfidf.fit_transform(games_with_content['content_text'])
item_to_tfidf = {int(row['item_idx']): i for i, (_, row) in enumerate(games_with_content.iterrows())}

hybrid_features = []
valid_items_11b = []

for item_idx in range(len(item_emb)):
    if item_idx not in item_to_tfidf:
        continue
    rs_emb      = item_emb[item_idx]
    tfidf_feat  = tfidf_matrix[item_to_tfidf[item_idx]].toarray().flatten()
    game_row    = games_with_content[games_with_content['item_idx'] == item_idx].iloc[0]
    price       = game_row['price_num']
    ea          = game_row['early_access_flag']
    dr          = dev_rep[item_idx]  # developer reputation
    combined    = np.concatenate([rs_emb, tfidf_feat, [price, ea, dr]])
    hybrid_features.append(combined)
    valid_items_11b.append(item_idx)

X_11b = np.array(hybrid_features)
print(f"X_11b shape: {X_11b.shape}  (RS 64d + TF-IDF 100d + Numeric 2d + dev_rep 1d)")

y_11b = target_df.set_index('item_idx').loc[valid_items_11b]['total_reviews'].values

data_df_11b = pd.DataFrame({'item_idx': valid_items_11b}).merge(
    df_games[['item_idx', 'release_date_parsed']],
    on='item_idx', how='left'
)

model_11b, r2_11b_test, r2_11b_tscv = run_experiment(
    X_11b, y_11b, data_df_11b,
    model_id="11b",
    model_name="Hybrid + Dev Reputation",
    features_desc="RS clean (64d) + TF-IDF (100d) + Numeric (2d) + dev_rep (1d)",
    emb_type="clean"
)

X_11b shape: (3195, 167)  (RS 64d + TF-IDF 100d + Numeric 2d + dev_rep 1d)

EXPERIMENT [11b]: Hybrid + Dev Reputation
Features: RS clean (64d) + TF-IDF (100d) + Numeric (2d) + dev_rep (1d)  |  X shape: (3195, 167)
Train: 2621, Test: 486


  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: 43.5659:   0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: 43.5659:   2%|▏         | 1/50 [00:00<00:37,  1.29it/s]

Best trial: 1. Best value: 9.33669:   2%|▏         | 1/50 [00:03<00:37,  1.29it/s]

...

Best trial: 47. Best value: 5.43077: 100%|██████████| 50/50 [01:23<00:00,  1.67s/it]

Best val RMSE: 5.4308


Temporal: R²=-0.016313 | RMSE=57.36 | MAE=9.40


TSCV:     R²=0.8580 ± 0.1613  RMSE=16.25


KFold:    R²=0.3073 ± 0.2583  RMSE=72.22
✅ Saved [11b]


In [5]:
# Final leaderboard and summary
import sys, os
sys.path.insert(0, os.path.abspath("."))
from results_tracker import print_leaderboard
print_leaderboard()

print("\n" + "="*70)
print("ABLATION: Developer Reputation Impact")
print("="*70)
print(f"03 RS Only          →  temporal R²: -0.0270 | TSCV R²: 0.7699")
print(f"11a RS + dev_rep    →  temporal R²: {r2_11a_test:+.4f} | TSCV R²: {r2_11a_tscv:.4f}")
print()
print(f"08 Hybrid           →  temporal R²: -0.0240 | TSCV R²: 0.8410")
print(f"11b Hybrid+dev_rep  →  temporal R²: {r2_11b_test:+.4f} | TSCV R²: {r2_11b_tscv:.4f}")
print("="*70)
print("\nNota: Un R² positivo en temporal para 11a/11b indicaría que dev_rep")
print("proporciona señal predictiva para juegos post-2016.")

 ID  Modelo                         R2 test     RMSE     MAE   MAPE%   R2 TSCV  R2 KFold
[13c]  Stage2 Content (log target)     0.2539     0.84    9.88     1.3    0.7570    0.7241
[12]  Two-Stage: Content Model        0.0760    54.69   12.22     2.8   -5.0275   -2.3051
[04]  Metadata Only                   0.0572    55.24   18.84     8.4   -0.2144    0.0710
[06]  Review Text Emb                -0.0107    57.20   18.76     8.3   -0.1720    0.0072
[05]  RS + Metadata                  -0.0161    57.35    9.16     0.6    0.8888    0.0110
[11b]  Hybrid + Dev Reputation        -0.0163    57.36    9.40     0.8    0.8580    0.3073
[10]  RS + Reviews + RAWG            -0.0163    57.36    9.14    46.6    0.8634    0.4330
[03]  RS Embeddings Only             -0.0184    57.42    9.64     1.1    0.8861    0.4510
[09]  RS + Review Text               -0.0213    57.50    9.27    49.1    0.8733    0.3359
[08]  Hybrid Collab-Content          -0.0240    57.57    9.37     0.4    0.8562    0.2945
[11a]  RS